In [2]:
import pandas as pd
from bertopic import BERTopic


cols = ["cord_uid", "title", "abstract", "publish_time",
        "authors", "journal", "license", "doi", "source_x"]

dtypes = {
    "cord_uid": "string", "title": "string", "abstract": "string",
    "authors": "string", "journal": "string", "license": "category",
    "doi": "string", "source_x": "category",
}

df = pd.read_csv("../data/metadata.csv", usecols=cols, dtype=dtypes, low_memory=False)
df["publish_time"] = pd.to_datetime(df["publish_time"], errors="coerce")
df = df.dropna(subset=["abstract"])


In [3]:
START, END = "2019-12-01", "2022-12-31"
df = df[(df["publish_time"] >= START) & (df["publish_time"] <= END)]
print(f"Rows in core pandemic window ({START} to {END}): {len(df)}")

Rows in core pandemic window (2019-12-01 to 2022-12-31): 375727


In [9]:
monthly = (
    df.set_index("publish_time")
      .resample("ME")
      .size()
      .rename("paper_count")
      .reset_index()
)
monthly.to_csv("../data/summary_monthly_volume.csv", index=False)
print(f"summary_monthly_volume.csv -> {len(monthly)} rows")


summary_monthly_volume.csv -> 37 rows


In [10]:
PREPRINT_MARKERS = ("biorxiv", "medrxiv", "arxiv", "preprint")

def classify_source(s):
    s = str(s).lower()
    return "Preprint" if any(m in s for m in PREPRINT_MARKERS) else "Peer-reviewed"

df["pub_type"] = df["source_x"].apply(classify_source)

pubtype_quarterly = (
    df.set_index("publish_time")
      .groupby([pd.Grouper(freq="QE"), "pub_type"])
      .size()
      .rename("paper_count")
      .reset_index()
      .rename(columns={"publish_time": "quarter"})
)
pubtype_quarterly["quarter"] = pubtype_quarterly["quarter"].dt.to_period("Q").astype(str)
pubtype_quarterly.to_csv("../data/summary_pubtype_quarterly.csv", index=False)
print(f"summary_pubtype_quarterly.csv -> {len(pubtype_quarterly)} rows")

summary_pubtype_quarterly.csv -> 24 rows


In [7]:
print("Unique license values in your data:", df["license"].astype(str).unique()[:20])

OPEN_LICENSES = {"cc0", "cc-by", "cc-by-nc", "cc-by-nc-sa", "cc-by-nd", "cc-by-sa", "pd", "gold-oa"}
df["is_open"] = df["license"].astype(str).str.lower().isin(OPEN_LICENSES)

license_yearly = (
    df.assign(year=df["publish_time"].dt.year)
      .groupby(["year", "is_open"])
      .size()
      .rename("paper_count")
      .reset_index()
)
license_yearly.to_csv("../data/summary_license_yearly.csv", index=False)
print(f"summary_license_yearly.csv -> {len(license_yearly)} rows")

Unique license values in your data: <ArrowStringArray>
['cc-by-nc-nd',       'no-cc', 'cc-by-nc-sa',       'cc-by',    'cc-by-nc',
         'cc0',   'hybrid-oa',     'gold-oa',    'green-oa',    'cc-by-nd',
         'unk',   'bronze-oa',    'cc-by-sa',   'els-covid',          'pd',
       'arxiv',     'biorxiv',     'medrxiv']
Length: 18, dtype: str
summary_license_yearly.csv -> 8 rows


In [ ]:
import re
from sklearn.feature_extraction.text import CountVectorizer

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_abstract"] = df["abstract"].apply(clean_text)


df_candidates = df[
    (df["clean_abstract"].str.split().str.len() > 30) &
    (~df["abstract"].str.contains("corrigendum|erratum|correction to", case=False, na=False))
]

sample_df = (
    df_candidates.dropna(subset=["clean_abstract"])
      .sample(15000, random_state=42)[["clean_abstract", "publish_time"]]
      .reset_index(drop=True)
)


custom_stopwords = ["also", "using", "used", "study", "results", "doi", "https", "com"]
vectorizer_model = CountVectorizer(
    stop_words=list(CountVectorizer(stop_words="english").get_stop_words()) + custom_stopwords,
    ngram_range=(1, 2),
    min_df=5
)

topic_model = BERTopic(language="english", min_topic_size=50, vectorizer_model=vectorizer_model, verbose=True)
topics, probs = topic_model.fit_transform(sample_df["clean_abstract"].tolist())
sample_df["topic"] = topics

topic_names = topic_model.get_topic_info().set_index("Topic")["Name"]
sample_df["topic_name"] = sample_df["topic"].map(topic_names)

top_topics = (
    sample_df[sample_df["topic"] != -1]["topic_name"]
    .value_counts()
    .head(6)
    .index
)
print("Top topics found:", list(top_topics))  

sample_top = sample_df[sample_df["topic_name"].isin(top_topics)].copy()
sample_top["quarter"] = sample_top["publish_time"].dt.to_period("Q").astype(str)

topic_quarterly = (
    sample_top.groupby(["quarter", "topic_name"])
              .size()
              .rename("paper_count")
              .reset_index()
)
topic_quarterly.to_csv("../data/summary_topic_quarterly.csv", index=False)
print(f"summary_topic_quarterly.csv -> {len(topic_quarterly)} rows")

2026-09-24 01:40:24,281 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

2026-09-24 02:00:13,345 - BERTopic - Embedding - Completed ✓
2026-09-24 02:00:13,348 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-24 02:01:46,636 - BERTopic - Dimensionality - Completed ✓
2026-09-24 02:01:46,639 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-24 02:01:50,134 - BERTopic - Cluster - Completed ✓
2026-09-24 02:01:50,185 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-24 02:02:11,681 - BERTopic - Representation - Completed ✓


Top topics found: ['0_anxiety_mental_mental health_health', '1_model_epidemic_cases_number', '2_cov_sars_sars cov_protein', '3_policy_health_economic_global', '4_students_learning_education_teaching', '5_care_telemedicine_telehealth_patient']
summary_topic_quarterly.csv -> 65 rows


In [11]:
browse_sample = (
    df[["title", "journal", "publish_time"]]
    .dropna(subset=["title"])
    .sample(2000, random_state=42)
    .sort_values("publish_time", ascending=False)
)
browse_sample.to_csv("../data/summary_browse_sample.csv", index=False)
print(f"summary_browse_sample.csv -> {len(browse_sample)} rows")

print("\nAll summary files written to ../data/. Total combined size should be well under 1MB.")
print("Record these numbers now — you'll need them for the README:")
print(f"  Total papers in core window: {len(df)}")
print(f"  Peak month: {monthly.loc[monthly['paper_count'].idxmax(), 'publish_time']} "
      f"with {monthly['paper_count'].max()} papers")

summary_browse_sample.csv -> 2000 rows

All summary files written to ../data/. Total combined size should be well under 1MB.
Record these numbers now — you'll need them for the README:
  Total papers in core window: 375727
  Peak month: 2022-03-31 00:00:00 with 17115 papers
